# 16 - Employee Intelligence Layer

**Objective:** Create one row per employee combining attrition risk, role, skill gaps, and recommendations.

This table is the actual business output of the whole project.

In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")

from app.ml.predictor import predict_employee

DATA_PATH = "../data/raw"
attrition = pd.read_csv(f"{DATA_PATH}/employee_attrition.csv")
print(f"Employees: {len(attrition)}")

Employees: 1470


In [2]:
from app.services.skill_gap_service import compute_employee_skill_gaps
from app.services.recommendation_service import generate_recommendations

# Build intelligence for a sample of 10 employees
sample = attrition.head(10)
results = []

for _, row in sample.iterrows():
    input_data = {
        "Age": int(row["Age"]), "Department": row["Department"],
        "DistanceFromHome": int(row["DistanceFromHome"]),
        "Education": int(row["Education"]), "EducationField": row["EducationField"],
        "EnvironmentSatisfaction": int(row["EnvironmentSatisfaction"]),
        "Gender": row["Gender"], "JobInvolvement": int(row["JobInvolvement"]),
        "JobLevel": int(row["JobLevel"]), "JobRole": row["JobRole"],
        "JobSatisfaction": int(row["JobSatisfaction"]),
        "MaritalStatus": row["MaritalStatus"],
        "MonthlyIncome": int(row["MonthlyIncome"]),
        "NumCompaniesWorked": int(row["NumCompaniesWorked"]),
        "OverTime": row["OverTime"],
        "PercentSalaryHike": int(row["PercentSalaryHike"]),
        "PerformanceRating": int(row["PerformanceRating"]),
        "RelationshipSatisfaction": int(row["RelationshipSatisfaction"]),
        "StockOptionLevel": int(row["StockOptionLevel"]),
        "TotalWorkingYears": int(row["TotalWorkingYears"]),
        "TrainingTimesLastYear": int(row["TrainingTimesLastYear"]),
        "WorkLifeBalance": int(row["WorkLifeBalance"]),
        "YearsAtCompany": int(row["YearsAtCompany"]),
        "YearsInCurrentRole": int(row["YearsInCurrentRole"]),
        "YearsSinceLastPromotion": int(row["YearsSinceLastPromotion"]),
        "YearsWithCurrManager": int(row["YearsWithCurrManager"]),
        "HourlyRate": int(row["HourlyRate"]),
        "DailyRate": int(row["DailyRate"]),
        "MonthlyRate": int(row["MonthlyRate"]),
        "BusinessTravel": row["BusinessTravel"],
    }
    pred = predict_employee(input_data)

    # Skill gaps for this employee's role (no per-employee skill data available)
    gaps = compute_employee_skill_gaps(
        [{"employee_id": str(row["EmployeeNumber"]), "role": row["JobRole"]}],
        {}
    )
    gap_info = gaps[0] if gaps else {}

    # Recommendations for missing skills
    recs = generate_recommendations(
        [{"employee_id": str(row["EmployeeNumber"]), "role": row["JobRole"]}],
        {}
    )
    rec_list = recs[0].get("recommendations", []) if recs else []

    results.append({
        "Employee_ID": str(row["EmployeeNumber"]),
        "Dept": row["Department"],
        "Role": row["JobRole"],
        "Attrition_Prob": pred["attrition_probability"],
        "Risk": pred["risk_level"],
        "Skill_Gaps": gap_info.get("gap_count", 0),
        "Recommendations": "; ".join(rec_list[:3]) if rec_list else "None",
        "Actual": row["Attrition"],
    })

df_intel = pd.DataFrame(results)
print("Employee Intelligence Table (sample):")
print(df_intel.to_string(index=False))

2026-09-01 17:25:37 | INFO | hr_ai.model | Loading attrition prediction model...


2026-09-01 17:25:43 | INFO | hr_ai.model | Model loaded: XGBoost vv1.0 (44 features)


Employee Intelligence Table (sample):
Employee_ID                   Dept                      Role  Attrition_Prob Risk  Skill_Gaps                                                                                                                                      Recommendations Actual
          1                  Sales           Sales Executive          0.9814 High          80 Review available training catalog for this skill; Review available training catalog for this skill; Review available training catalog for this skill    Yes
          2 Research & Development        Research Scientist          0.0143  Low          47 Review available training catalog for this skill; Review available training catalog for this skill; Review available training catalog for this skill     No
          4 Research & Development     Laboratory Technician          0.9799 High          29 Review available training catalog for this skill; Review available training catalog for this skill; Review available t

In [3]:
# Summary
print(f"High risk: {(df_intel['Risk'] == 'High').sum()}")
print(f"Medium risk: {(df_intel['Risk'] == 'Medium').sum()}")
print(f"Low risk: {(df_intel['Risk'] == 'Low').sum()}")
print(f"Actual attrition in sample: {(df_intel['Actual'] == 'Yes').sum()}/{len(df_intel)}")

High risk: 2
Medium risk: 0
Low risk: 8
Actual attrition in sample: 2/10


## Findings

- The Employee Intelligence Layer combines attrition prediction with employee context
- Each row provides: who they are, their risk level, prediction probability, skill gaps, and upskilling recommendations
- Skill gaps are computed by comparing role requirements (essential_skills + software_skills) against assumed zero current skills
- Recommendations map each missing skill to a training course via rule-based matching
- The API endpoint serves this intelligence on demand